# Analysis of Inverse Convolution Functions $G^{-1} * F$ and $F^{-1} * G$

In this notebook, we analyze the behavior of the convolution of inverses of the left and right augmented Chow functions ($G$ and $F$).

## Definitions

*   **Chow polynomial** ($H_M(x)$): The polynomial associated with the Chow ring of a matroid.
*   **Reverse Kazhdan-Lusztig polynomial** ($f^{\text{rev}}_M(x)$): $P_M^!(x) = x^{\text{rk} M} P_M(x^{-1})$.
*   **Right augmented Chow function** ($F$): $F = H * f^{\text{rev}}$.
*   **Left augmented Chow function** ($G$): $G = f^{\text{rev}} * H$.

The convolution $(A * B)(M)$ is defined as $\sum_{F \in \mathcal{L}(M)} A(M|F) B(M/F)$.

In [32]:
import sys
import os

# Setup paths
current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)
sys.path.insert(0, parent_dir)

from sage.all import *
from custom_matroid_functions import *

P.<x> = LaurentPolynomialRing(ZZ)

def chow_poly_fn(M):
    return P(chow_polynomial(M))

def kl_rev_fn(M):
    p = kl_poly(M)
    t = p.parent().gen()
    rk = M.rank()
    return P(p.subs({t: x**-1}) * x**rk)

def get_matroid_key(M):
    rank = M.rank()
    size = len(M.groundset())
    try:
        can = M.canonical_form()
        return (rank, size, str(can))
    except AttributeError:
        bases = tuple(sorted(tuple(sorted(b)) for b in M.bases()))
        return (rank, size, str(bases))

def convolution_inverse(f):
    cache_inv = {}
    
    def f_inv(M):
        m_key = get_matroid_key(M)
        if m_key in cache_inv:
            return cache_inv[m_key]
        
        if M.rank() == 0 and M.size() == 0:
            res = 1 / f(M)
            cache_inv[m_key] = res
            return res
        
        # f_inv(M) = - (1/f(empty)) * sum_{empty != F <= E} f(M|F) f_inv(M/F)
        empty_matroid = matroids.Uniform(0, 0)
        f0 = f(empty_matroid)
        
        L = M.lattice_of_flats()
        E = M.groundset()
        
        total = P(0)
        for F in L:
            if len(F) == 0: continue
            restriction = M.delete(list(set(E) - set(F)))
            contraction = M.contract(list(F))
            total += f(restriction) * f_inv(contraction)
            
        res = -total / f0
        cache_inv[m_key] = res
        return res
    
    return f_inv

chow = chow_polynomial
chibar = reduced_characteristic_polynomial
char_poly = characteristic_polynomial
kl_poly = lambda M: M.lattice_of_flats().kazhdan_lusztig_polynomial()(x)
grev = lambda M: P(x**M.rank())
mu_rev = lambda M: P(x**M.rank() * chibar(M)(0))
mu = lambda M: P(char_poly(M)(0))
F_fn = convolve(chow_poly_fn, kl_rev_fn)
G_fn = convolve(kl_rev_fn, chow_poly_fn)

G_inv_fn = convolve(chibar, mu_rev)

G_inv_star_F = convolve(mu_rev, F_fn)

## Analysis for Rank 3 Matroids

We test several rank 3 matroids to observe the pattern of $(G^{-1} * F)(M)$.

In [33]:
matroids_to_test = [
    (matroids.Uniform(3, 4), "U(3, 4)"),
    (matroids.Uniform(3, 5), "U(3, 5)"),
    (matroids.Uniform(3, 6), "U(3, 6)"),
    (matroids.CompleteGraphic(4), "K4")
]

for M, label in matroids_to_test:
    print(f"{label}: {G_inv_star_F(M)}")

U(3, 4): -1 - 9*x - 3*x^2
U(3, 5): -1 - 16*x - 6*x^2
U(3, 6): -1 - 25*x - 10*x^2
K4: -1 - 9*x - 2*x^2


## Analysis for Rank 4 Matroids

We test rank 4 uniform matroids.

In [37]:
matroids_to_test = [
    (matroids.Uniform(5, 5), "U(5, 5)"),
    (matroids.Uniform(6, 6), "U(6, 6)"),
    (matroids.Uniform(4, 7), "U(4, 7)")
]

for M, label in matroids_to_test:
    print(f"{label}: {G_inv_star_F(M)}")

print(chow_polynomial(matroids_to_test[1][0]))

U(5, 5): -1 - 26*x - 66*x^2 - 26*x^3 - x^4
U(6, 6): -1 - 57*x - 302*x^2 - 302*x^3 - 57*x^4 - x^5
U(4, 7): -1 - 85*x - 85*x^2 + 34*x^3
x^5 + 57*x^4 + 302*x^3 + 302*x^2 + 57*x + 1


## Conclusions

1. **Symmetry**: In all cases, $F^{-1} * G = - (G^{-1} * F)$.
2. **Linear Pattern (Rank 3)**: For rank 3 uniform matroids $U_{3,n}$, $G^{-1} * F (M) = \frac{n(n-3)}{2} x$. 
3. **Quadratic Pattern (Rank 4)**: For rank 4 uniform matroids $U_{4,n}$, $G^{-1} * F (M)$ is a quadratic in $x$.

In [35]:
matroids_to_test = [
    (matroids.CompleteGraphic(5), "U(5, 5)"),
    (matroids.CompleteGraphic(4), "U(4, 7)")
]

for M, label in matroids_to_test:
    print(f"{label}: {G_inv_star_F(M)}")

U(5, 5): -1 - 46*x - 51*x^2 + 4*x^3
U(4, 7): -1 - 9*x - 2*x^2
